# 02 — Open-Set Scorer: Mahalanobis vs Confidence (Paper 3, Tahap T3)

**Dijalankan di SageMaker.** Menjawab: *bagaimana model tahu sebuah flow adalah jenis
BARU (unknown), bukan salah satu kelas known?* Memakai artefak notebook **01**:
model known + `deploy_meta_mc_<DS>.json` (scaler + centroid `mu` + `inv_cov` per-kelas).

**Alur:**
1. Muat model known + meta (mu/inv_cov) dari `01` (lokal `known_base_out/` atau S3).
2. Muat data: kelas **known** (test split) vs kelas **held-out** ("serangan baru").
3. Hitung dua skor open-set per flow:
   - **Mahalanobis (utama):** $d(x)=\min_k \sqrt{(x-\mu_k)^\top \Sigma_k^{-1}(x-\mu_k)}$.
   - **Confidence (pembanding):** $1-\max_k p_k(x)$ (makin tinggi = makin unknown).
4. Ukur **AUROC known-vs-unknown** (label: known=0, held-out=1) untuk kedua skor.
5. Kalibrasi ambang $\tau$ dari distribusi jarak known (persentil-99) → laporkan
   TPR(held-out ketangkap) & FPR(known salah-tandai) pada $\tau$ itu.

**Output** (→ S3 `evolusion/openset/`): `openset_results.json`, plot distribusi skor
known vs held-out, ROC curve.

> Catatan kejujuran (documentation.md §4): XGBoost sering **overconfident** pada input
> tak-dikenal, jadi confidence bisa lemah. Notebook ini **menguji**, bukan mengasumsikan;
> Mahalanobis dijadikan basis. Semua angka dari eksekusi nyata.

In [ ]:
import importlib.util as u, sys, subprocess
need=[m for m in ('xgboost','scikit-learn','scipy','pandas','numpy','matplotlib','boto3') if u.find_spec(m.replace('scikit-learn','sklearn')) is None]
if need: subprocess.run([sys.executable,'-m','pip','install','-q',*need],check=True)
print('setup ok' if not need else f'installed {need}')
print('=== SEL 0 (setup) SELESAI ===')

In [ ]:
import os, json, glob, datetime
import numpy as np, pandas as pd
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb
plt.rcParams.update({'figure.dpi':120,'font.size':9})
S3_BUCKET=os.environ.get('S3_BUCKET','ssh-detection-features-232032302717')
S3_PREFIX='evolusion'; REGION=os.environ.get('AWS_REGION','ap-southeast-1')
IN01='known_base_out'          # folder output notebook 01 (lokal)
OUTDIR='openset_out'; os.makedirs(OUTDIR,exist_ok=True)
CANON=['duration','fwd_pkts','bwd_pkts','fwd_bytes','bwd_bytes','fwd_mean','bwd_mean','src_load','dst_load']
SEED=42; MIN_CLASS=200
# HELDOUT harus SAMA dgn notebook 01 (kelas yg disembunyikan saat latih):
HELDOUT={'CIC':['Botnet','Infiltration'],'UNSW':['Worms','Shellcode','Backdoor']}
RESULTS={'generated':datetime.datetime.utcnow().isoformat()+'Z','features':CANON,'seed':SEED,'heldout':HELDOUT}
def savefig(n): p=os.path.join(OUTDIR,n); plt.savefig(p,bbox_inches='tight'); plt.close(); print(' saved',p); return p
def first(paths):
    for p in paths:
        h=sorted(glob.glob(p))
        if h: return h[0]
    return None

def fetch_from_s3_if_missing(fname):
    """Jika artefak 01 tak ada lokal, coba unduh dari S3 evolusion/known_base/."""
    lp=os.path.join(IN01,fname)
    if os.path.exists(lp): return lp
    try:
        import boto3; os.makedirs(IN01,exist_ok=True)
        boto3.client('s3',region_name=REGION).download_file(S3_BUCKET,f'{S3_PREFIX}/known_base/{fname}',lp)
        print('   diunduh dari S3:',fname); return lp
    except Exception as e:
        print('   GAGAL ambil',fname,'->',e); return None
print('=== SEL 1 (config) SELESAI ===')

## 2. Loader data + mapping label (identik notebook 01)

In [ ]:
def map_cic(lbl):
    s=str(lbl).strip().lower()
    if s in ('benign','normal'): return 'Benign'
    if s.startswith('ddos') or 'loic' in s or 'hoic' in s: return 'DDoS'
    if s.startswith('dos'): return 'DoS'
    if 'bruteforce' in s or 'brute force' in s or 'ftp-brute' in s or 'ssh-brute' in s: return 'BruteForce'
    if s=='bot' or 'botnet' in s: return 'Botnet'
    if 'infil' in s: return 'Infiltration'
    if 'web' in s or 'xss' in s or 'sql' in s: return 'Web'
    return 'Other'
def map_uns(lbl):
    s=str(lbl).strip().lower()
    if s in ('normal','benign',''): return 'Benign'
    return {'dos':'DoS','exploits':'Exploits','fuzzers':'Fuzzers','generic':'Generic',
            'reconnaissance':'Recon','backdoor':'Backdoor','backdoors':'Backdoor',
            'shellcode':'Shellcode','worms':'Worms','analysis':'Analysis'}.get(s,'Other')

def load_cic():
    p=first(['../../CICDDoS2018/data/file_100.csv','../../CICDDoS2018/data/file_*.csv'])
    if not p: print('CIC csv tak ada'); return None
    c=pd.read_csv(p, low_memory=False); c.columns=c.columns.str.strip()
    cm={'duration':'Flow Duration','fwd_pkts':'Tot Fwd Pkts','bwd_pkts':'Tot Bwd Pkts','fwd_bytes':'TotLen Fwd Pkts',
        'bwd_bytes':'TotLen Bwd Pkts','fwd_mean':'Fwd Pkt Len Mean','bwd_mean':'Bwd Pkt Len Mean','src_load':'Flow Byts/s','dst_load':'Bwd Pkts/s'}
    lc=[x for x in c.columns if x.lower()=='label']; LAB=lc[0] if lc else c.columns[-1]
    if not all(v in c.columns for v in cm.values()): print('CIC kolom kurang'); return None
    d=pd.DataFrame({k:pd.to_numeric(c[cm[k]],errors='coerce') for k in CANON})
    d['cat']=c[LAB].map(map_cic)
    d=d.replace([np.inf,-np.inf],np.nan).dropna(); d=d[d['cat']!='Other']
    return d

def _pick_unsw_train():
    cands=[]
    for pat in ['../data/UNSW_NB15_*set.csv','../../unswnb-15/data/UNSW_NB15_*set.csv']:
        cands+=sorted(glob.glob(pat))
    cands=list(dict.fromkeys(cands))
    if not cands: return None
    best,best_n=None,-1
    for p in cands:
        try: n=sum(1 for _ in open(p,'r',errors='ignore'))-1
        except Exception: n=-1
        if n>best_n: best,best_n=p,n
    print(f'    UNSW dipakai: {os.path.basename(best)} (~{best_n} record)')
    return best

def load_uns():
    p=_pick_unsw_train()
    if not p: print('UNSW csv tak ada'); return None
    u2=pd.read_csv(p); need=['dur','spkts','dpkts','sbytes','dbytes','smean','dmean','sload','dload','attack_cat']
    if not all(x in u2.columns for x in need): print('UNSW kolom kurang'); return None
    d=pd.DataFrame({'duration':pd.to_numeric(u2['dur'],errors='coerce')*1e6,'fwd_pkts':u2['spkts'],'bwd_pkts':u2['dpkts'],
                    'fwd_bytes':u2['sbytes'],'bwd_bytes':u2['dbytes'],'fwd_mean':u2['smean'],'bwd_mean':u2['dmean'],
                    'src_load':pd.to_numeric(u2['sload'],errors='coerce')/8.0,
                    'dst_load':pd.to_numeric(u2['dpkts'],errors='coerce')/pd.to_numeric(u2['dur'],errors='coerce').replace(0,np.nan)})
    d['cat']=u2['attack_cat'].fillna('Normal').map(map_uns)
    d=d.replace([np.inf,-np.inf],np.nan).dropna(); d=d[d['cat']!='Other']
    return d

cic=load_cic(); uns=load_uns()
print('=== SEL 2 (loader) SELESAI ===')

## 3. Skor open-set: Mahalanobis + confidence

In [ ]:
def load_artifacts(ds):
    mp=fetch_from_s3_if_missing(f'deploy_meta_mc_{ds}.json')
    md=fetch_from_s3_if_missing(f'model_known_{ds}.json')
    if not mp or not md: return None
    meta=json.load(open(mp))
    booster=xgb.Booster(); booster.load_model(md)
    mean=np.asarray(meta['scaler_mean'],float); scale=np.asarray(meta['scaler_scale'],float)
    labels=meta['labels']
    mus=np.array([meta['class_stats'][c]['mu'] for c in labels],float)          # [K,9]
    icovs=np.array([meta['class_stats'][c]['inv_cov'] for c in labels],float)    # [K,9,9]
    return dict(meta=meta,booster=booster,mean=mean,scale=scale,labels=labels,mus=mus,icovs=icovs)

def scale_X(art, X):
    return np.nan_to_num((X-art['mean'])/art['scale'], nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)

def mahalanobis_min(art, Xs):
    """Jarak Mahalanobis minimum ke centroid kelas known (di ruang ter-scale). Return [N]."""
    mus, icovs = art['mus'], art['icovs']  # [K,9], [K,9,9]
    N=Xs.shape[0]; K=mus.shape[0]
    dmin=np.full(N, np.inf)
    for k in range(K):
        diff=Xs-mus[k]                      # [N,9]
        # d2 = sum((diff @ icov) * diff, axis=1)
        d2=np.einsum('ni,ij,nj->n', diff, icovs[k], diff)
        d2=np.clip(d2, 0, None)
        dmin=np.minimum(dmin, np.sqrt(d2))
    return dmin

def confidence_unknown(art, Xs):
    """Skor unknown berbasis confidence = 1 - max prob. Return [N]."""
    prob=art['booster'].predict(xgb.DMatrix(Xs))
    if prob.ndim==1: prob=np.vstack([1-prob,prob]).T   # jaga-jaga biner
    return 1.0 - prob.max(axis=1)
print('=== SEL 3 (fungsi skor open-set) SELESAI ===')

## 4. Evaluasi per dataset: known (test) vs held-out ("baru")

In [ ]:
def merge_rare(df, min_n=MIN_CLASS):
    vc=df['cat'].value_counts(); rare=[c for c,n in vc.items() if n<min_n and c!='Benign']
    if rare:
        df=df.copy(); df['cat']=df['cat'].where(~df['cat'].isin(rare),'Other-rare')
    return df

def eval_openset(ds, df):
    if df is None: print(f'[{ds}] data kosong, lewati'); return None
    art=load_artifacts(ds)
    if art is None: print(f'[{ds}] artefak 01 tak ada, jalankan notebook 01 dulu'); return None
    heldout=HELDOUT.get(ds,[])
    known_labels=art['labels']
    # KNOWN: rekonstruksi test split IDENTIK notebook 01 (kelas known, merge_rare, split 0.3 seed 42)
    dk=merge_rare(df[~df['cat'].isin(heldout)].copy())
    dk=dk[dk['cat'].isin(known_labels)]  # samakan label set dgn model
    le=LabelEncoder().fit(known_labels)
    Xk=dk[CANON].values; yk=le.transform(dk['cat'].values)
    _,Xk_te,_,_=train_test_split(Xk,yk,test_size=0.3,random_state=SEED,stratify=yk)
    # HELD-OUT: seluruh flow kelas held-out = 'unknown' sejati
    dh=df[df['cat'].isin(heldout)].copy()
    Xh=dh[CANON].values
    if len(Xh)==0: print(f'[{ds}] tak ada flow held-out (cek HELDOUT vs data)'); return None
    # skor
    Xk_s=scale_X(art,Xk_te); Xh_s=scale_X(art,Xh)
    maha_k=mahalanobis_min(art,Xk_s); maha_h=mahalanobis_min(art,Xh_s)
    conf_k=confidence_unknown(art,Xk_s); conf_h=confidence_unknown(art,Xh_s)
    # label: known=0, unknown(held-out)=1
    y=np.r_[np.zeros(len(maha_k)), np.ones(len(maha_h))]
    auroc_maha=float(roc_auc_score(y, np.r_[maha_k,maha_h]))
    auroc_conf=float(roc_auc_score(y, np.r_[conf_k,conf_h]))
    # kalibrasi tau = persentil-99 jarak known -> TPR/FPR pada tau
    tau=float(np.percentile(maha_k,99))
    tpr=float((maha_h>tau).mean())   # held-out ketangkap sbg unknown
    fpr=float((maha_k>tau).mean())   # known salah-tandai unknown (~1%)
    res=dict(dataset=ds, known_classes=known_labels, heldout_classes=heldout,
             n_known_test=int(len(maha_k)), n_heldout=int(len(maha_h)),
             auroc_mahalanobis=round(auroc_maha,4), auroc_confidence=round(auroc_conf,4),
             tau_p99_known=round(tau,4), tpr_heldout_at_tau=round(tpr,4), fpr_known_at_tau=round(fpr,4))
    print(f"[{ds}] AUROC maha={res['auroc_mahalanobis']} conf={res['auroc_confidence']} | "
          f"tau(p99)={res['tau_p99_known']} TPR_heldout={res['tpr_heldout_at_tau']} FPR_known={res['fpr_known_at_tau']}")
    # plot distribusi skor Mahalanobis known vs held-out + ROC
    fig,ax=plt.subplots(1,2,figsize=(11,4))
    ax[0].hist(maha_k,bins=60,alpha=0.6,label='known',density=True,color='#4C72B0')
    ax[0].hist(maha_h,bins=60,alpha=0.6,label='held-out (baru)',density=True,color='#C44E52')
    ax[0].axvline(tau,ls='--',color='k',label=f'tau(p99)={tau:.1f}')
    ax[0].set_xlabel('Mahalanobis min-dist'); ax[0].set_ylabel('densitas'); ax[0].legend(fontsize=8)
    ax[0].set_title(f'{ds}: distribusi skor Mahalanobis')
    for score_k,score_h,nm,c in [(maha_k,maha_h,'Mahalanobis','#4C72B0'),(conf_k,conf_h,'Confidence','#DD8452')]:
        fp,tp,_=roc_curve(y,np.r_[score_k,score_h]); a=roc_auc_score(y,np.r_[score_k,score_h])
        ax[1].plot(fp,tp,color=c,label=f'{nm} (AUROC={a:.3f})')
    ax[1].plot([0,1],[0,1],':',color='gray'); ax[1].set_xlabel('FPR'); ax[1].set_ylabel('TPR')
    ax[1].set_title(f'{ds}: ROC known-vs-unknown'); ax[1].legend(fontsize=8)
    plt.tight_layout(); savefig(f'openset_{ds}.png')
    return res

RESULTS['openset']={}
for ds,df in [('CIC',cic),('UNSW',uns)]:
    r=eval_openset(ds,df)
    if r: RESULTS['openset'][ds]=r
print('=== SEL 4 (evaluasi open-set) SELESAI ===')

## 5. Ringkas + simpan + UPLOAD S3

In [ ]:
rows=[]
for ds,r in RESULTS.get('openset',{}).items():
    rows.append({'dataset':ds,'held_out':','.join(r['heldout_classes']),
                 'AUROC_maha':r['auroc_mahalanobis'],'AUROC_conf':r['auroc_confidence'],
                 'tau_p99':r['tau_p99_known'],'TPR_heldout':r['tpr_heldout_at_tau'],'FPR_known':r['fpr_known_at_tau']})
summ=pd.DataFrame(rows)
import IPython.display as ipd; print('Ringkasan open-set (Mahalanobis vs Confidence):'); ipd.display(summ)
summ.to_csv(os.path.join(OUTDIR,'openset_summary.csv'),index=False)
jp=os.path.join(OUTDIR,'openset_results.json'); json.dump(RESULTS,open(jp,'w'),indent=2); print('tersimpan',jp)
try:
    import boto3; s3=boto3.client('s3',region_name=REGION); up=0
    for fn in sorted(os.listdir(OUTDIR)):
        if fn.endswith(('.json','.png','.csv')): s3.upload_file(os.path.join(OUTDIR,fn),S3_BUCKET,f'{S3_PREFIX}/openset/{fn}'); up+=1
    print(f'upload {up} artefak -> s3://{S3_BUCKET}/{S3_PREFIX}/openset/')
except Exception as e: print('upload gagal:',e)
print('=== SEL 5 (simpan + upload) SELESAI ===')
print('SELESAI T3. Interpretasi: AUROC maha tinggi -> open-set memisahkan known vs baru. '
      'Flow di atas tau = kandidat unknown -> masuk notebook 03 (novelty clustering).')